# Entrenamiento HMM - ETTh1 para varios K
Entrena Baum-Welch con K = {5, 10, 15, 20} sobre datos RevIN-normalizados (per-window) y guarda cache para cada K.

In [1]:
import numpy as np
import pandas as pd
import torch
import sys
import time
sys.path.insert(0, '..')
from hmm.baum_welch import baum_welch

In [2]:
# Cargar train set y normalizar con RevIN per-window
df = pd.read_csv('../dataset/ETT-small/ETTh1.csv')
n_train = int(len(df) * 0.7)
train_ot = df['OT'].values[:n_train]

seq_len = 96
windows = []
for start in range(0, len(train_ot) - seq_len + 1, seq_len):
    window = train_ot[start:start + seq_len]
    w_mean, w_std = window.mean(), max(window.std(), 1e-5)
    windows.append((window - w_mean) / w_std)

train_norm = np.concatenate(windows)
print(f'Train: {len(train_norm)} timesteps ({len(windows)} ventanas x {seq_len})')
print(f'RevIN-normalized: mean={train_norm.mean():.6f}, std={train_norm.std():.6f}')

Train: 12192 timesteps (127 ventanas x 96)
RevIN-normalized: mean=-0.000000, std=1.000000


In [3]:
# Entrenar y cachear para cada K (skip si ya existe en cache)
import os
K_values = [5, 10, 15, 20]

for K in K_values:
    path = f'../cache/hmm_etth1_K{K}.pth'

    if os.path.exists(path):
        c = torch.load(path, weights_only=False)
        if c['converged']:
            print(f'K={K}: CACHE EXISTENTE (convergido) - iter {c["n_iter"]}, LL={c["log_likelihood"]:.2f}')
            continue
        else:
            print(f'K={K}: cache existe pero NO convergio - reentrenando...')

    print(f'\n{"="*60}')
    print(f'Entrenando K={K}')
    print(f'{"="*60}')

    t0 = time.time()
    result = baum_welch(
        observations=train_norm,
        K=K,
        max_iter=2000,
        epsilon=1e-4,
        random_state=42,
        verbose=True
    )
    elapsed = time.time() - t0

    # Validar
    assert result['converged'], f'K={K} NO CONVERGIO en {result["n_iter"]} iters'
    assert result['log_likelihood'] != float('-inf'), f'K={K} log_likelihood es -inf'
    assert np.all(result['sigma'] > 0), f'K={K} sigma tiene valores <= 0'
    assert np.allclose(result['A'].sum(axis=1), 1.0, atol=1e-6), f'K={K} filas de A no suman 1'

    # Guardar cache
    cache = {
        'A': torch.from_numpy(result['A']).float(),
        'pi': torch.from_numpy(result['pi']).float(),
        'mu': torch.from_numpy(result['mu']).float(),
        'sigma': torch.from_numpy(result['sigma']).float(),
        'log_likelihood': result['log_likelihood'],
        'converged': result['converged'],
        'n_iter': result['n_iter'],
    }
    torch.save(cache, path)

    print(f'\nK={K}: converged iter {result["n_iter"]}, LL={result["log_likelihood"]:.2f}, tiempo={elapsed:.1f}s')
    print(f'  mu: {result["mu"]}')
    print(f'  sigma: {result["sigma"]}')
    print(f'  Cache guardado: {path}')

K=5: CACHE EXISTENTE (convergido) - iter 108, LL=-8631.79
K=10: CACHE EXISTENTE (convergido) - iter 205, LL=-6722.06
K=15: CACHE EXISTENTE (convergido) - iter 607, LL=-6095.69

Entrenando K=20


Baum-Welch EM:  28%|██▊       | 551/2000 [5:04:02<13:19:32, 33.11s/it, LL=-5864.31, ΔLL=1.01e-04]



Convergió en iteración 552/2000
  Log-likelihood final: -5864.3148

K=20: converged iter 552, LL=-5864.31, tiempo=18243.2s
  mu: [-1.75684304  0.72590927 -0.28986969  1.21471531 -0.80077774  0.11819909
  1.49418682 -1.48057125 -2.35336609  0.50546712  2.33809949 -0.09252859
  1.85980513  0.98101268 -0.60657986 -1.27314044 -0.45476398 -2.12050215
 -1.05591394  0.31091539]
  sigma: [0.16036304 0.09646027 0.07701817 0.09661592 0.0868983  0.09273351
 0.13626117 0.09966788 0.53310848 0.08825821 0.36768409 0.0778645
 0.18413139 0.09456756 0.06506816 0.08414977 0.05913799 0.1782955
 0.09665976 0.07988408]
  Cache guardado: ../cache/hmm_etth1_K20.pth


In [4]:
# Resumen comparativo
print(f'{"K":>3} | {"Iters":>5} | {"LL":>12} | {"sigma_min":>10} | {"sigma_max":>10} | {"A_diag_mean":>11}')
print('-' * 70)
for K in K_values:
    c = torch.load(f'../cache/hmm_etth1_K{K}.pth', weights_only=False)
    sigma = c['sigma'].numpy()
    A_diag = torch.diag(c['A']).numpy()
    print(f'{K:3d} | {c["n_iter"]:5d} | {c["log_likelihood"]:12.2f} | {sigma.min():10.4f} | {sigma.max():10.4f} | {A_diag.mean():11.4f}')

  K | Iters |           LL |  sigma_min |  sigma_max | A_diag_mean
----------------------------------------------------------------------
  5 |   108 |     -8631.79 |     0.2322 |     0.4834 |      0.7872
 10 |   205 |     -6722.06 |     0.1407 |     0.4605 |      0.6109
 15 |   607 |     -6095.69 |     0.0858 |     0.6312 |      0.5045
 20 |   552 |     -5864.31 |     0.0591 |     0.5331 |      0.4136
